# 📊 Étape 4 : Visualisation Multidimensionnelle (Squelette Étudiant)

Cette étape correspond au quatrième chapitre du cours. L'objectif est de concevoir des représentations visuelles pour identifier des tendances et insights clés du dataset **Automobile** : évolution de la consommation par année, corrélations entre variables, et relations bivariées entre puissance et efficacité.

### 1. Préparation de l'environnement

In [15]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import pandas as pd
import plotly.graph_objects as go
import os

sys.path.append(os.path.abspath('..'))

print("Librairies de visualisation prêtes !")

Librairies de visualisation prêtes !


### 2. Chargement du dataset enrichi

In [16]:
df = pd.read_csv('../data/processed/cleaned_data_sample.csv')

# Recréation des features issues de l'EDA
df['power_to_weight'] = df['horsepower'] / df['weight']
df['mpg_category'] = pd.qcut(
    df['mpg'],
    q=3,
    labels=['faible', 'moyenne', 'élevée']
)

print(f"Shape : {df.shape}")
df.head()

Shape : (398, 10)


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,power_to_weight,mpg_category
0,18.0,8,307.0,130.0,3504,12.0,70,2,0.037100,faible
1,15.0,8,350.0,165.0,3693,11.5,70,2,0.044679,faible
2,18.0,8,318.0,150.0,3436,11.0,70,2,0.043655,faible
3,16.0,8,304.0,150.0,3433,12.0,70,2,0.043694,faible
4,17.0,8,302.0,140.0,3449,10.5,70,2,0.040591,faible


### 3. Tracés et analyses graphiques

#### A. Évolution des tendances dans le temps

Évolution de la consommation moyenne (MPG) par année de modèle — permet d'observer le progrès des motorisations au fil du temps.

In [17]:

# 1. Chargement des données (C'EST LA LIGNE QUI MANQUAIT !)
df = pd.read_csv('../data/processed/cleaned_data_sample.csv')

# 2. Calcul de la moyenne par année 
mpg_per_year = df.groupby('model_year')['mpg'].mean()

# 3. Création du graphique interactif en ligne
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=mpg_per_year.index,
    y=mpg_per_year.values,
    mode='lines+markers',       # On garde la ligne et les points
    name='MPG Moyen',
    marker=dict(size=8),        # Taille des points
    line=dict(width=3)          # Épaisseur de la ligne
))

# 4. Mise en page globale (Titres et Thème)
fig.update_layout(
    title_text="<b>Évolution de la consommation moyenne (MPG) par année de modèle</b>",
    title_font_size=16,
    xaxis_title="Année du modèle (model_year)",
    yaxis_title="MPG moyen",
    template="plotly_white",     # Le thème sombre
    height=500,
    hovermode="x unified"       # Bulle info qui suit la souris verticalement
)

# 5. Sauvegarde en format Web Interactif (.html)
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)
html_path = os.path.join(output_dir, 'mpg_evolution.html')
fig.write_html(html_path)

print(f"Graphique interactif sauvegardé : {html_path}")

# Affichage interactif dans le Notebook
fig.show()

Graphique interactif sauvegardé : ../data/processed\mpg_evolution.html


#### B. Carte de chaleur des corrélations

Heatmap de la matrice de corrélation de Pearson sur l'ensemble des variables numériques, incluant la variable dérivée `power_to_weight`.

In [18]:

# 1. Vos colonnes souhaitées
corr_cols = ['mpg', 'cylinders', 'displacement', 'horsepower',
             'weight', 'acceleration', 'model_year', 'origin', 'power_to_weight']

# SÉCURITÉ : On ne garde que les colonnes qui existent actuellement dans votre variable 'df'
existing_cols = [c for c in corr_cols if c in df.columns]

# 2. Calcul de la matrice de corrélation
corr_matrix = df[existing_cols].corr(method='pearson')

# 3. Création de la Heatmap interactive avec Plotly
fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.index,
    colorscale='RdBu_r',                  
    zmin=-1, zmax=1,                      
    text=np.round(corr_matrix.values, 2), 
    texttemplate='%{text}',               
    hoverinfo='x+y+z'                     
))

# 4. Mise en page globale
fig.update_layout(
    title_text="<b>Carte de chaleur des corrélations de Pearson — Dataset Automobile</b>",
    title_font_size=16,
    template="plotly_white",               
    width=800,
    height=800,
    xaxis_showgrid=False,
    yaxis_showgrid=False,
    yaxis_autorange='reversed'            
)

# 5. Sauvegarde en format Web Interactif (.html)
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)
html_path = os.path.join(output_dir, 'correlation_heatmap.html')
fig.write_html(html_path)

print(f"Graphique interactif sauvegardé : {html_path}")

# Affichage interactif dans le Notebook
fig.show()

Graphique interactif sauvegardé : ../data/processed\correlation_heatmap.html


#### C. Nuage de points bivarié

Relation entre la puissance moteur (`horsepower`) et la consommation (`mpg`), colorée par origine géographique du véhicule (0=Europe, 1=Japon, 2=USA).

In [19]:

# 1. Chargement des données (Pour éviter l'erreur "df is not defined")
df = pd.read_csv('../data/processed/cleaned_data_sample.csv')

# Configuration des origines avec des couleurs adaptées au thème sombre
origin_config = {
    0: ('Europe', '#3b82f6'), # Bleu
    1: ('Japan',  '#ef4444'), # Rouge
    2: ('USA',    '#22c55e')  # Vert
}

# 2. Création du graphique interactif
fig = go.Figure()

for origin_code, (label, color) in origin_config.items():
    # Sécurité : vérifier que l'origine existe bien dans les données
    if origin_code in df['origin'].unique():
        subset = df[df['origin'] == origin_code]
        
        fig.add_trace(go.Scatter(
            x=subset['horsepower'],
            y=subset['mpg'],
            mode='markers',
            name=label,
            marker=dict(
                color=color,
                size=10,                      # L'équivalent de s=55 dans Matplotlib
                opacity=0.7,                  # L'équivalent de alpha=0.6
                line=dict(width=1, color='white') # L'équivalent de edgecolors
            ),
            # Ce qui s'affiche quand la souris survole le point
            hovertemplate="<b>" + label + "</b><br>Puissance : %{x} hp<br>Consommation : %{y} MPG<extra></extra>"
        ))

# 3. Mise en page globale (Titres et Thème)
fig.update_layout(
    title_text="<b>Nuage de points : Puissance vs Consommation (MPG) par Origine</b>",
    title_font_size=16,
    xaxis_title="Puissance (Horsepower)",
    yaxis_title="Consommation (MPG)",
    legend_title_text="Origine",
    template="plotly_white",     # Le thème sombre pour votre dashboard
    height=600,
    hovermode="closest"         # Met en évidence le point le plus proche de la souris
)

# 4. Sauvegarde en format Web Interactif (.html) au lieu du .png
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)
html_path = os.path.join(output_dir, 'scatter_hp_mpg.html')
fig.write_html(html_path)

print(f"Graphique interactif sauvegardé : {html_path}")

# Affichage interactif dans le Notebook
fig.show()

Graphique interactif sauvegardé : ../data/processed\scatter_hp_mpg.html
